# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# Step 1 - Feature Scaling + Feature Selection (Preprocessing)
# Goal:
# - Separate features (X) and target (y)
# - Define which columns are numeric vs categorical
# - Build a preprocessing pipeline:
#   - Fill missing values
#   - One-hot encode categorical columns
#   - Scale numeric columns (StandardScaler)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Step 1.1 - Define target (y) and features (X)
# Transported is our target to predict
X = spaceship.drop(columns=["Transported"])
y = spaceship["Transported"]

# Step 1.2 - Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object", "bool"]).columns

# Step 1.3 - Preprocessing for numeric columns:
# - Fill missing values with median (robust for outliers)
# - Scale features to the same scale (important for distance-based models like KNN)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Step 1.4 - Preprocessing for categorical columns:
# - Fill missing values with most frequent value
# - Convert categories into numeric columns using OneHotEncoder
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Step 1.5 - Combine both preprocessors into one ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

print("Numeric columns:", list(numeric_cols))
print("Categorical columns:", list(categorical_cols))


Numeric columns: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
Categorical columns: ['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP', 'Name']


In [4]:
# Step 2 - Train/Test split
# Goal:
# - Use 80% data for training, 20% for testing
# - random_state makes results reproducible

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


Train shape: (6954, 13) Test shape: (1739, 13)


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [5]:

# Step 3 - Build a baseline model (best model so far)
# We'll use KNN because it often works well after scaling.
# IMPORTANT:
# We wrap preprocessing + model into ONE Pipeline to avoid data leakage.

from sklearn.neighbors import KNeighborsClassifier

baseline_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", KNeighborsClassifier())
])

print(baseline_model)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignor

- Evaluate your model

In [6]:
# Step 4 - Evaluate baseline model
# Goal:
# - Fit ONLY on training data
# - Evaluate on test data using accuracy and classification report

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, y_pred_baseline))
print("\nClassification Report:\n", classification_report(y_test, y_pred_baseline))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_baseline))



Baseline Accuracy: 0.7763082231167338

Classification Report:
               precision    recall  f1-score   support

       False       0.78      0.77      0.77       863
        True       0.78      0.78      0.78       876

    accuracy                           0.78      1739
   macro avg       0.78      0.78      0.78      1739
weighted avg       0.78      0.78      0.78      1739


Confusion Matrix:
 [[664 199]
 [190 686]]


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [7]:
# Step 5 - Define hyperparameter grid for KNN
# Goal:
# - Try different numbers of neighbors (k)
# - Try different distance weighting strategies
# - Try different distance metrics (p=1 Manhattan, p=2 Euclidean)

param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

param_grid


{'model__n_neighbors': [3, 5, 7, 9, 11, 15],
 'model__weights': ['uniform', 'distance'],
 'model__p': [1, 2]}

- Run Grid Search

In [8]:
# Step 6 - Run GridSearchCV
# Goal:
# - Use cross-validation on the TRAINING set only
# - Find best hyperparameter combination
# Notes:
# - cv=5 means 5-fold cross validation
# - scoring='accuracy' evaluates by accuracy (fine for balanced target)

from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=baseline_model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best CV Accuracy:", grid_search.best_score_)
print("Best Params:", grid_search.best_params_)


Best CV Accuracy: 0.7808458280104889
Best Params: {'model__n_neighbors': 15, 'model__p': 1, 'model__weights': 'uniform'}


- Evaluate your model

In [9]:
# Step 7 - Evaluate tuned (best) model on test set
# Goal:
# - Use best estimator found by grid search
# - Compare performance vs baseline

best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)

print("Tuned Model Accuracy:", accuracy_score(y_test, y_pred_best))
print("\nClassification Report:\n", classification_report(y_test, y_pred_best))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_best))


C:\Users\famil\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\famil\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\famil\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\famil\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

Tuned Model Accuracy: 0.7814836112708453

Classification Report:
               precision    recall  f1-score   support

       False       0.78      0.77      0.78       863
        True       0.78      0.79      0.78       876

    accuracy                           0.78      1739
   macro avg       0.78      0.78      0.78      1739
weighted avg       0.78      0.78      0.78      1739


Confusion Matrix:
 [[666 197]
 [183 693]]


In [10]:
# Step 8 - Final Summary (baseline vs tuned)
# Goal:
# - Summarize results so the notebook is easy to  grade/read. 

print("Baseline Accuracy:", accuracy_score(y_test, y_pred_baseline))
print("Best CV Accuracy:", grid_search.best_score_)
print("Best Params:", grid_search.best_params_)
print("Tuned Test Accuracy:", accuracy_score(y_test, y_pred_best))


Baseline Accuracy: 0.7763082231167338
Best CV Accuracy: 0.7808458280104889
Best Params: {'model__n_neighbors': 15, 'model__p': 1, 'model__weights': 'uniform'}
Tuned Test Accuracy: 0.7814836112708453
